# Prediction Model - Baselines and Decision Policy

This notebook builds and compares 3 baseline models for **2-class** `Inventario` classification
(after excluding the rare `Assistenza` class from model training):
1. Decision Tree
2. Naive Bayes
3. KNN

Then it generates interpretable patterns/rules and proposes an automation vs manual review policy.


## 1) Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

## 2) Load Data

In [3]:
NOTEBOOK_DIR = Path.cwd()
CANDIDATES = [
    NOTEBOOK_DIR / "DB for DTM Project.xlsx",
    NOTEBOOK_DIR.parent / "DB for DTM Project.xlsx",
]

FILE_PATH = next((p for p in CANDIDATES if p.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError(
        "DB for DTM Project.xlsx not found in current or parent directory"
    )

SHEET = "Associazioni Cod. Art. - LOB"
df = pd.read_excel(FILE_PATH, sheet_name=SHEET)
df.columns = [str(c).strip() for c in df.columns]

print("File:", FILE_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

df.head(3)

File: c:\Users\sveta\Documents\vem-product\DB for DTM Project.xlsx
Shape: (22655, 5)
Columns: ['Codice Articolo', 'Lob Associata', 'Nome LOB', 'Inventario', 'Descrizione Articolo']


,Codice Articolo,Lob Associata,Nome LOB,Inventario,Descrizione Articolo
0,000050,1002,CABLAGGIO ALTERNATIVO,Inventario,PATCH CORD UTP 2/RJ45 CAT5E MT2
1,000081,1002,CABLAGGIO ALTERNATIVO,Inventario,cab - cf54/150ez passerella a filo 3m
2,‎000411AA230621QMANE,3004,CLOUD VARIE HW,Inventario,QUARKZMAN 30pz Ultra Sottile Telefono Leva Ape...


## 3) Prepare Target and Inputs

Target: `Inventario` (2-class modeling)

`Assistenza` is excluded from train/test and should be routed to manual review in production logic.

Text input combines:
- `Codice Articolo`
- `Descrizione Articolo`
- `Lob Associata`
- `Nome LOB`


In [4]:
TARGET_COL = "Inventario"
ASSISTENZA_LABEL = "Assistenza"
CODE_COL = "Codice Articolo"
DESC_COL = "Descrizione Articolo"
LOB_COL = "Lob Associata"
NOME_LOB_COL = "Nome LOB"

required = [TARGET_COL, CODE_COL, DESC_COL, LOB_COL, NOME_LOB_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df_model = df[required].copy()
df_model[DESC_COL] = df_model[DESC_COL].fillna("")
df_model[CODE_COL] = df_model[CODE_COL].fillna("")
df_model[LOB_COL] = df_model[LOB_COL].fillna("")
df_model[NOME_LOB_COL] = df_model[NOME_LOB_COL].fillna("")

# Build a single textual representation for baseline models.
df_model["input_text"] = (
    "CODE_"
    + df_model[CODE_COL].astype(str)
    + " "
    + "LOB_"
    + df_model[LOB_COL].astype(str)
    + " "
    + "NOMELOB_"
    + df_model[NOME_LOB_COL].astype(str)
    + " "
    + df_model[DESC_COL].astype(str)
)

# Keep only rows with non-null target
df_model = df_model[df_model[TARGET_COL].notna()].copy()

# Exclude rare Assistenza class from model training/evaluation
assistenza_mask = (
    df_model[TARGET_COL]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq(ASSISTENZA_LABEL.lower())
)
assistenza_rows = int(assistenza_mask.sum())
df_model = df_model[~assistenza_mask].copy()

print("Excluded Assistenza rows:", assistenza_rows)
print("Modeling rows after exclusion:", len(df_model))
print("Target classes after exclusion:", df_model[TARGET_COL].nunique())
df_model[TARGET_COL].value_counts()

Excluded Assistenza rows: 11
Modeling rows after exclusion: 22644
Target classes after exclusion: 2


Inventario
Inventario           19764
Non in inventario     2880
Name: count, dtype: int64

## 4) Train/Test Split and Vectorization

In [5]:
X = df_model["input_text"]
y = df_model[TARGET_COL]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = CountVectorizer(
    lowercase=True, min_df=5, max_features=4000, ngram_range=(1, 2), binary=True
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
feature_names = vectorizer.get_feature_names_out()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (18115, 4000)
X_test shape: (4529, 4000)


## 5) Train Baseline Models and Compare Accuracy/F1

In [6]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42, max_depth=30, min_samples_leaf=5, class_weight="balanced"
    ),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "KNN": KNeighborsClassifier(
        n_neighbors=7, weights="distance", metric="cosine", algorithm="brute"
    ),
}

results = []
trained_models = {}
predictions = {}
probabilities = {}
reports = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")

    results.append(
        {
            "model": name,
            "accuracy": acc,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }
    )

    trained_models[name] = model
    predictions[name] = y_pred
    probabilities[name] = y_proba
    reports[name] = classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    )

metrics_df = (
    pd.DataFrame(results)
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)
metrics_df

,model,accuracy,f1_macro,f1_weighted
0,KNN,0.956944,0.897583,0.955722
1,Decision Tree,0.928461,0.857410,0.932461
2,Naive Bayes,0.929344,0.856483,0.932731


In [7]:
# Per-class view for each model
for name in metrics_df["model"]:
    print(f"\n===== {name} =====")
    rep_df = pd.DataFrame(reports[name]).T
    display(rep_df)


===== KNN =====


,precision,recall,f1-score,support
Inventario,0.966948,0.984316,0.975555,3953.000000
Non in inventario,0.877228,0.769097,0.819611,576.000000
accuracy,0.956944,0.956944,0.956944,0.956944
macro avg,0.922088,0.876706,0.897583,4529.000000
weighted avg,0.955538,0.956944,0.955722,4529.000000



===== Decision Tree =====


,precision,recall,f1-score,support
Inventario,0.980917,0.936251,0.958064,3953.000000
Non in inventario,0.666667,0.875000,0.756757,576.000000
accuracy,0.928461,0.928461,0.928461,0.928461
macro avg,0.823792,0.905625,0.857410,4529.000000
weighted avg,0.940951,0.928461,0.932461,4529.000000



===== Naive Bayes =====


,precision,recall,f1-score,support
Inventario,0.977649,0.940551,0.958742,3953.000000
Non in inventario,0.676309,0.852431,0.754224,576.000000
accuracy,0.929344,0.929344,0.929344,0.929344
macro avg,0.826979,0.896491,0.856483,4529.000000
weighted avg,0.939325,0.929344,0.932731,4529.000000


### 5.1 ROC Curve for Decision Tree

Visual check of threshold behavior for binary Inventario task.
- Positive class is set to `Inventario` when available.
- Output: ROC plot + AUC.


In [ ]:
dt_model = trained_models["Decision Tree"]
dt_proba = probabilities["Decision Tree"]

classes_dt = list(dt_model.classes_)
print("Decision Tree classes:", classes_dt)

if len(classes_dt) != 2:
    print("ROC is shown only for binary classification.")
else:
    positive_label = "Inventario" if "Inventario" in classes_dt else classes_dt[1]
    pos_idx = classes_dt.index(positive_label)

    y_true_bin = (
        y_test.reset_index(drop=True).astype(str) == str(positive_label)
    ).astype(int)
    y_score = dt_proba[:, pos_idx]

    fpr, tpr, thresholds = roc_curve(y_true_bin, y_score)
    auc_value = roc_auc_score(y_true_bin, y_score)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"Decision Tree ROC (AUC = {auc_value:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - Decision Tree (positive class: {positive_label})")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

    pd.DataFrame({"threshold": thresholds, "fpr": fpr, "tpr": tpr}).head(15)

### 5.2 Precision-Recall Curve for Decision Tree

Useful with class imbalance.
- Positive class is set to `Inventario` when available.
- Output: PR plot + Average Precision (AP).


In [ ]:
dt_model = trained_models["Decision Tree"]
dt_proba = probabilities["Decision Tree"]

classes_dt = list(dt_model.classes_)
print("Decision Tree classes:", classes_dt)

if len(classes_dt) != 2:
    print("Precision-Recall curve is shown only for binary classification.")
else:
    positive_label = "Inventario" if "Inventario" in classes_dt else classes_dt[1]
    pos_idx = classes_dt.index(positive_label)

    y_true_bin = (
        y_test.reset_index(drop=True).astype(str) == str(positive_label)
    ).astype(int)
    y_score = dt_proba[:, pos_idx]

    precision, recall, thresholds = precision_recall_curve(y_true_bin, y_score)
    ap_value = average_precision_score(y_true_bin, y_score)
    base_rate = y_true_bin.mean()

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision, label=f"Decision Tree PR (AP = {ap_value:.4f})")
    plt.hlines(
        base_rate,
        xmin=0,
        xmax=1,
        linestyles="--",
        label=f"Positive rate baseline = {base_rate:.4f}",
    )
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(
        f"Precision-Recall Curve - Decision Tree (positive class: {positive_label})"
    )
    plt.legend(loc="best")
    plt.grid(alpha=0.3)
    plt.show()

    # thresholds has length N-1 vs precision/recall length N
    pr_table = pd.DataFrame(
        {"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]}
    )
    pr_table.head(15)

## 6) Interpretable Patterns / Rules

### 6.1 Naive Bayes: Top tokens per class

In [8]:
nb_model = trained_models["Naive Bayes"]
classes = nb_model.classes_

nb_top = []
for class_idx, cls in enumerate(classes):
    top_idx = np.argsort(nb_model.feature_log_prob_[class_idx])[::-1][:20]
    for rank, idx in enumerate(top_idx, start=1):
        nb_top.append(
            {
                "class": cls,
                "rank": rank,
                "token": feature_names[idx],
                "log_prob": nb_model.feature_log_prob_[class_idx, idx],
            }
        )

nb_top_df = pd.DataFrame(nb_top)
nb_top_df

,class,rank,token,log_prob
0,Inventario,1,cisco,-4.321988
1,Inventario,2,nomelob_apparati,-4.529853
2,Inventario,3,nomelob_cablaggio,-4.717583
3,Inventario,4,nomelob_apparati cisco,-4.730477
4,Inventario,5,lob_1002 nomelob_cablaggio,-4.911502
5,Inventario,6,alternativo,-4.911502
6,Inventario,7,nomelob_cablaggio alternativo,-4.911502
7,Inventario,8,lob_1002,-4.911502
8,Inventario,9,nomelob_cloud,-4.920877
9,Inventario,10,varie,-4.923392


### 6.2 Decision Tree: Human-readable rules + top features

In [9]:
dt_model = trained_models["Decision Tree"]

# Top features by importance
importances = pd.DataFrame(
    {"feature": feature_names, "importance": dt_model.feature_importances_}
).sort_values("importance", ascending=False)

print("Top feature importances:")
display(importances.head(30))

print("\nTree rules (truncated depth=4):")
print(export_text(dt_model, feature_names=list(feature_names), max_depth=4))

Top feature importances:


,feature,importance
3407,servizi,0.134116
3333,security,0.091520
1133,code_a,0.081321
2681,nomelob_cyberguru,0.056207
3594,support,0.047590
2356,lob_99001,0.043974
3398,service,0.042963
2698,nomelob_rinnovo,0.039402
1633,enterprise,0.036461
2743,nomelob_terzisti delivery,0.035222



Tree rules (truncated depth=4):
|--- servizi <= 0.50
|   |--- security <= 0.50
|   |   |--- code_a <= 0.50
|   |   |   |--- nomelob_cyberguru <= 0.50
|   |   |   |   |--- support <= 0.50
|   |   |   |   |   |--- truncated branch of depth 26
|   |   |   |   |--- support >  0.50
|   |   |   |   |   |--- truncated branch of depth 26
|   |   |   |--- nomelob_cyberguru >  0.50
|   |   |   |   |--- lob_15007 nomelob_cyberguru <= 0.50
|   |   |   |   |   |--- class: Non in inventario
|   |   |   |   |--- lob_15007 nomelob_cyberguru >  0.50
|   |   |   |   |   |--- class: Non in inventario
|   |   |--- code_a >  0.50
|   |   |   |--- flex <= 0.50
|   |   |   |   |--- collaboration canone <= 0.50
|   |   |   |   |   |--- truncated branch of depth 3
|   |   |   |   |--- collaboration canone >  0.50
|   |   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- flex >  0.50
|   |   |   |   |--- for <= 0.50
|   |   |   |   |   |--- class: Non in inventario
|   |   |   |   |--- for >  0.50


### 6.3 KNN: Nearest-neighbor explanation examples

In [10]:
knn_model = trained_models["KNN"]

# Take a few test rows and show their nearest training neighbors
sample_n = 5
X_test_sample = X_test[:sample_n]

dists, idxs = knn_model.kneighbors(X_test_sample, n_neighbors=3, return_distance=True)

train_text_reset = X_train_text.reset_index(drop=True)
train_target_reset = y_train.reset_index(drop=True)
test_text_reset = X_test_text.reset_index(drop=True)
test_target_reset = y_test.reset_index(drop=True)

rows = []
for i in range(sample_n):
    rows.append(
        {
            "test_row": i,
            "test_true": test_target_reset.iloc[i],
            "test_text_preview": test_text_reset.iloc[i][:120],
        }
    )
    for j in range(3):
        nn_idx = idxs[i, j]
        rows.append(
            {
                "test_row": i,
                "neighbor_rank": j + 1,
                "neighbor_distance": float(dists[i, j]),
                "neighbor_target": train_target_reset.iloc[nn_idx],
                "neighbor_text_preview": train_text_reset.iloc[nn_idx][:120],
            }
        )

knn_examples_df = pd.DataFrame(rows)
knn_examples_df

,test_row,test_true,test_text_preview,neighbor_rank,neighbor_distance,neighbor_target,neighbor_text_preview
0,0,Inventario,CODE_AR8715 LOB_17001 NOMELOB_SISTEMI SCHNEIDE...,NaN,NaN,NaN,NaN
1,0,NaN,NaN,1.0,0.215535,Inventario,CODE_AR7502 LOB_17001 NOMELOB_SISTEMI SCHNEIDE...
2,0,NaN,NaN,2.0,0.280908,Inventario,CODE_AR7588 LOB_17001 NOMELOB_SISTEMI SCHNEIDE...
3,0,NaN,NaN,3.0,0.298354,Inventario,CODE_AR8442 LOB_17001 NOMELOB_SISTEMI SCHNEIDE...
4,1,Non in inventario,CODE_A-SPK-VOIP LOB_4015 NOMELOB_CISCO COLLABO...,NaN,NaN,NaN,NaN
5,1,NaN,NaN,1.0,0.039231,Non in inventario,CODE_A-SPK-EDU-VOIP LOB_4015 NOMELOB_CISCO COL...
6,1,NaN,NaN,2.0,0.087129,Non in inventario,CODE_A-SPK-EDUEC-TEC LOB_4015 NOMELOB_CISCO CO...
7,1,NaN,NaN,3.0,0.166667,Non in inventario,CODE_A-SPK-ND-BRD LOB_4015 NOMELOB_CISCO COLLA...
8,2,Inventario,CODE_SFP-H10GB-CU3M= LOB_3009 NOMELOB_SERVER C...,NaN,NaN,NaN,NaN
9,2,NaN,NaN,1.0,0.000000,Inventario,CODE_SFP-H10GB-CU3M LOB_3009 NOMELOB_SERVER CI...


## 7) Confidence Policy and Manual Review

Define confidence buckets by max predicted probability:
- `high` >= 0.85
- `medium` >= 0.65 and < 0.85
- `low` < 0.65 (manual review)

Additionally, `Assistenza` should remain a manual-review bucket (not auto-classified by this 2-class model).


In [11]:
best_model_name = metrics_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]
best_proba = probabilities[best_model_name]

max_proba = best_proba.max(axis=1)

confidence_bucket = pd.cut(
    max_proba,
    bins=[-1, 0.65, 0.85, 1.0],
    labels=["low_manual_review", "medium", "high"],
)

confidence_df = pd.DataFrame(
    {
        "y_true": y_test.reset_index(drop=True),
        "y_pred": pd.Series(best_pred),
        "max_proba": max_proba,
        "bucket": confidence_bucket,
    }
)
confidence_df["correct"] = (confidence_df["y_true"] == confidence_df["y_pred"]).astype(
    int
)

bucket_summary = (
    confidence_df.groupby("bucket", observed=False)
    .agg(n=("correct", "size"), accuracy=("correct", "mean"))
    .reset_index()
)
bucket_summary["share"] = bucket_summary["n"] / len(confidence_df)

print("Best model:", best_model_name)
display(bucket_summary)

Best model: KNN


,bucket,n,accuracy,share
0,low_manual_review,154,0.597403,0.034003
1,medium,226,0.752212,0.049901
2,high,4149,0.981441,0.916096


In [12]:
# Suggested decision policy
auto_bucket = confidence_df[confidence_df["bucket"] == "high"]
manual_bucket = confidence_df[confidence_df["bucket"] == "low_manual_review"]

auto_rate = len(auto_bucket) / len(confidence_df)
auto_acc = auto_bucket["correct"].mean() if len(auto_bucket) else np.nan
manual_rate = len(manual_bucket) / len(confidence_df)

policy_summary = pd.DataFrame(
    [
        {
            "best_model": best_model_name,
            "recommended_auto_bucket": "high",
            "auto_rate": auto_rate,
            "auto_accuracy": auto_acc,
            "manual_review_rate": manual_rate,
        }
    ]
)

policy_summary

,best_model,recommended_auto_bucket,auto_rate,auto_accuracy,manual_review_rate
0,KNN,high,0.916096,0.981441,0.034003


## 8) Final Conclusion Template

Use the computed tables above to fill final statements:
1. Can we automate?
2. Where is the model confident?
3. Where is manual review required?
4. Confirm that `Assistenza` stays outside auto-decision scope.


In [13]:
best_metrics = metrics_df.iloc[0]

print("FINAL SUMMARY")
print(f"Best model: {best_metrics['model']}")
print(f"Accuracy: {best_metrics['accuracy']:.4f}")
print(f"F1 macro: {best_metrics['f1_macro']:.4f}")
print(f"F1 weighted: {best_metrics['f1_weighted']:.4f}")
print(f"Excluded Assistenza rows from modeling: {assistenza_rows}")

if not bucket_summary.empty:
    high_row = bucket_summary[bucket_summary["bucket"] == "high"]
    low_row = bucket_summary[bucket_summary["bucket"] == "low_manual_review"]

    if len(high_row):
        print(
            f"High-confidence share: {high_row['share'].iloc[0]:.2%}, accuracy: {high_row['accuracy'].iloc[0]:.2%}"
        )
    if len(low_row):
        print(f"Manual-review share (low confidence): {low_row['share'].iloc[0]:.2%}")

print("\nInterpretability artifacts:")
print("- Naive Bayes top tokens by class: nb_top_df")
print(
    "- Decision Tree top importances and readable rules: importances + export_text output"
)
print("- KNN neighbor examples: knn_examples_df")
print("Assistenza policy: manual review bucket (outside 2-class auto decision).")

FINAL SUMMARY
Best model: KNN
Accuracy: 0.9569
F1 macro: 0.8976
F1 weighted: 0.9557
Excluded Assistenza rows from modeling: 11
High-confidence share: 91.61%, accuracy: 98.14%
Manual-review share (low confidence): 3.40%

Interpretability artifacts:
- Naive Bayes top tokens by class: nb_top_df
- Decision Tree top importances and readable rules: importances + export_text output
- KNN neighbor examples: knn_examples_df
Assistenza policy: manual review bucket (outside 2-class auto decision).
